# TikTok search-term discovery

This notebook does one thing: it starts with an initial seed term and discovers the search language TikTok associates with it.

It follows the layered-expansion pattern used in `google_suggestions.ipynb`:

```text
seed term
Ã¢â€ â€™ first-order TikTok suggestions
Ã¢â€ â€™ second-order suggestions
Ã¢â€ â€™ optional third-order suggestions
```

The notebook uses a visible Chrome session and only reads suggestions and related-search terms shown by TikTok. It does not collect video URLs, score trends, download content, or use private APIs.

In [ ]:
# Install missing notebook dependencies. The installed Google Chrome is used,
# so Playwright does not need to download a separate browser.
import importlib.util
import subprocess
import sys

requirements = {
    "playwright": "playwright>=1.47,<2",
    "pandas": "pandas>=2.2,<3",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Dependencies are already installed.")

## 1. Configuration

Edit `SEARCH_TERM`. The final workflow uses exactly two expansion layers so every second-layer suggestion remains traceable to its first-layer parent without uncontrolled topic drift.

In [ ]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from playwright.async_api import async_playwright

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROFILE_DIR = PROJECT_ROOT / ".chrome_profile_search_terms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_TERM = "creepytok"
MAX_DEPTH = 2                 # fixed two-level discovery contract
MAX_SUGGESTIONS_PER_QUERY = 12
MAX_PARENTS_PER_DEPTH = 20   # prevents combinatorial expansion
MAX_TOTAL_TERMS = 200
QUERY_PAUSE_MS = 900
INCLUDE_RELATED_SEARCHES = False  # keep discovery limited to TikTok autocomplete
HEADLESS = False             # visible browser permits manual login/challenges

print("Seed term:", SEARCH_TERM)
print("Expansion depth:", MAX_DEPTH)

In [ ]:
UI_LABELS = {
    "accounts", "discover", "explore", "for you", "hashtags",
    "live", "search", "sounds", "users", "videos", "view more",
}

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def normalize_term(value: str) -> str:
    value = (value or "").replace("#", " " )
    value = re.sub(r"\s+", " ", value).strip(" -\u2013\u2014|:;,.")
    if not 2 <= len(value) <= 80 or "\n" in value:
        return ""
    if value.casefold() in UI_LABELS:
        return ""
    return value

def unique_terms(values: list[str], exclude: set[str] | None = None) -> list[str]:
    exclude = {value.casefold() for value in (exclude or set())}
    seen = set(exclude)
    output = []
    for value in values:
        term = normalize_term(value)
        key = term.casefold()
        if term and key not in seen:
            seen.add(key)
            output.append(term)
    return output

def save_json(path: Path, payload: Any) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

assert normalize_term("  #BookTok  " ) == "BookTok"
assert unique_terms(["BookTok", "booktok", "Book reviews"]) == ["BookTok", "Book reviews"]
print("Helper tests passed.")

## 2. Open TikTok in a persistent Chrome profile

Sign in manually if required. If TikTok presents a verification challenge, complete it in the visible browser; the notebook does not bypass challenges.

In [ ]:
playwright_instance = await async_playwright().start()
browser_context = await playwright_instance.chromium.launch_persistent_context(
    user_data_dir=str(PROFILE_DIR),
    channel="chrome",
    headless=HEADLESS,
    viewport=None,
    locale="en-AU",
    args=["--start-maximized"],
)
page = browser_context.pages[0] if browser_context.pages else await browser_context.new_page()
await page.goto("https://www.tiktok.com/", wait_until="domcontentloaded", timeout=90_000)
print("Chrome is open. Sign in or resolve any visible challenge before continuing.")

## 3. TikTok suggestion collector

The function below reads visible autocomplete entries. Optionally, it submits the query and also reads related-search links displayed by TikTok. Each discovered term retains its source.

In [ ]:
EXTRACT_VISIBLE_TERMS_JS = r"""
() => {
  const selectors = [
    '[role="option"]',
    '[data-e2e*="suggest"]',
    '[data-e2e*="search-item"]'
  ];
  const visible = (element) => Boolean(
    element.offsetWidth || element.offsetHeight || element.getClientRects().length
  );
  const values = [];
  for (const element of document.querySelectorAll(selectors.join(','))) {
    if (!visible(element)) continue;
    const text = (element.innerText || element.textContent || '').trim();
    if (text) values.push(text);
  }
  return values;
}
"""

EXTRACT_RELATED_SEARCHES_JS = r"""
() => {
  const values = [];
  for (const anchor of document.querySelectorAll('a[href*="/search"]')) {
    if (!(anchor.offsetWidth || anchor.offsetHeight || anchor.getClientRects().length)) continue;
    const text = (anchor.innerText || anchor.textContent || '').trim();
    if (text) values.push(text);
    try {
      const query = new URL(anchor.href).searchParams.get('q');
      if (query) values.push(query);
    } catch (_) {}
  }
  return values;
}
"""

async def find_search_input(page, timeout_ms: int = 15_000):
    # TikTok sometimes renders a search box and sometimes only a Search button.
    # Poll visible candidates, opening the Search control once if necessary.
    selectors = (
        'input[data-e2e="search-user-input"]',
        'input[data-e2e*="search"]',
        'input[type="search"]',
        'input[placeholder*="Search" i]',
    )
    search_opened = False
    attempts = max(1, timeout_ms // 500)
    for _ in range(attempts):
        for selector in selectors:
            candidates = page.locator(selector)
            for index in range(await candidates.count()):
                candidate = candidates.nth(index)
                if await candidate.is_visible():
                    return candidate
        if not search_opened:
            for trigger_selector in (
                'button[data-e2e="nav-search"]',
                'button[aria-label="Search"]',
            ):
                trigger = page.locator(trigger_selector).first
                if await trigger.count() and await trigger.is_visible():
                    await trigger.click()
                    search_opened = True
                    break
        await page.wait_for_timeout(500)
    raise RuntimeError(
        f"TikTok search input was not visible after {timeout_ms / 1000:.0f}s. "
        f"Current URL: {page.url}"
    )

TIKTOK_SUGGESTION_ENDPOINT = "https://www.tiktok.com/api/search/general/sug/"

async def get_tiktok_suggestions(query: str) -> list[dict[str, str]]:
    query = normalize_term(query)
    if not query:
        return []

    # This is the JSON endpoint used by TikTok's own web autocomplete. It is
    # simpler and less brittle than scraping the rendered suggestion panel.
    response = await page.request.get(
        TIKTOK_SUGGESTION_ENDPOINT,
        params={"keyword": query, "aid": "1988"},
        headers={"Referer": "https://www.tiktok.com/"},
        timeout=30_000,
    )
    if not response.ok:
        raise RuntimeError(f"TikTok suggestion request failed: HTTP {response.status}")
    payload = await response.json()
    autocomplete = unique_terms(
        [item.get("content", "") for item in payload.get("sug_list", [])],
        exclude={query},
    )[:MAX_SUGGESTIONS_PER_QUERY]
    await page.wait_for_timeout(QUERY_PAUSE_MS)
    return [{"term": term, "source": "tiktok_autocomplete"} for term in autocomplete]

print("Suggestion collector ready.")

## 4. Expand the seed term

This is a bounded breadth-first expansion. Terms remain separated by order, and every child retains its parent query and source.

In [ ]:
if not SEARCH_TERM or "REPLACE WITH" in SEARCH_TERM.upper():
    raise ValueError("Edit SEARCH_TERM in the configuration cell first.")
if MAX_DEPTH != 2:
    raise ValueError("The final workflow requires exactly two search-term layers.")

seed = normalize_term(SEARCH_TERM)
seen = {seed.casefold()}
frontier = [seed]
layers: dict[int, list[str]] = defaultdict(list)
records: list[dict[str, Any]] = []
edges: list[dict[str, str | int]] = []
term_paths: dict[str, list[str]] = {seed.casefold(): [seed]}

for depth in range(1, MAX_DEPTH + 1):
    next_frontier = []
    parents = frontier[:MAX_PARENTS_PER_DEPTH]
    print(f"\nExpanding order {depth}: {len(parents)} parent term(s)")
    for parent_index, parent in enumerate(parents, start=1):
        print(f"  {parent_index}/{len(parents)} {parent!r}", end="\r")
        suggestions = await get_tiktok_suggestions(parent)
        for suggestion in suggestions:
            child = suggestion["term"]
            key = child.casefold()
            parent_path = term_paths.get(parent.casefold(), [seed, parent])
            child_path = [*parent_path, child]
            first_order_term = child if depth == 1 else child_path[1]
            edges.append(
                {
                    "parent": parent,
                    "child": child,
                    "depth": depth,
                    "first_order_term": first_order_term,
                    "query_path": " > ".join(child_path),
                    "source": suggestion["source"],
                }
            )
            if key in seen:
                continue
            seen.add(key)
            term_paths[key] = child_path
            layers[depth].append(child)
            next_frontier.append(child)
            records.append(
                {
                    "term": child,
                    "order": depth,
                    "first_order_term": first_order_term,
                    "parent": parent,
                    "query_path": " > ".join(child_path),
                    "source": suggestion["source"],
                    "review_decision": "unreviewed",
                    "review_notes": "",
                    "collected_at": utc_now_iso(),
                }
            )
            if len(seen) >= MAX_TOTAL_TERMS:
                break
        if len(seen) >= MAX_TOTAL_TERMS:
            break
    print(f"\nOrder {depth}: {len(layers[depth])} new unique terms")
    frontier = next_frontier
    if not frontier or len(seen) >= MAX_TOTAL_TERMS:
        break

terms_df = pd.DataFrame(
    records,
    columns=[
        "term", "order", "first_order_term", "parent", "query_path",
        "source", "review_decision", "review_notes", "collected_at",
    ],
)
terms_for_review_df = terms_df.sort_values(
    ["first_order_term", "order", "term"],
    key=lambda column: column.astype(str).str.casefold(),
).reset_index(drop=True)
terms_for_review_df.insert(
    0,
    "display_term",
    terms_for_review_df.apply(
        lambda row: row["term"] if int(row["order"]) == 1 else f"    -> {row['term']}",
        axis=1,
    ),
)
terms_for_review_df.insert(
    0,
    "layer",
    terms_for_review_df["order"].map({1: "FIRST_LAYER", 2: "SECOND_LAYER", 3: "THIRD_LAYER"}),
)
print(f"Total unique discovered terms: {len(terms_df)}")

In [ ]:
branch_summary_df = (
    terms_df.groupby("first_order_term", sort=False)
    .agg(
        second_layer_terms=("order", lambda values: int((values == 2).sum())),
        total_terms=("term", "count"),
    )
    .reset_index()
)
display(branch_summary_df)
display(terms_for_review_df.head(15))

## 5. Export search terms

The CSV is the human review surface. The JSON preserves the nested branch structure for the next notebook.

In [ ]:
latest_csv_path = OUTPUT_DIR / "latest_search_terms.csv"
latest_json_path = OUTPUT_DIR / "latest_search_terms.json"

terms_for_review_df.to_csv(latest_csv_path, index=False, encoding="utf-8-sig")
branches = [
    {
        "first_order_term": first_order_term,
        "review_decision": "unreviewed",
        "review_notes": "",
        "first_layer": next(
            record for record in records
            if record["order"] == 1
            and record["term"].casefold() == first_order_term.casefold()
        ),
        "second_layer": [
            record for record in records
            if record["order"] == 2
            and record["first_order_term"].casefold() == first_order_term.casefold()
        ],
        "deeper_layers": [
            record for record in records
            if record["order"] > 2
            and record["first_order_term"].casefold() == first_order_term.casefold()
        ],
    }
    for first_order_term in layers.get(1, [])
]
payload = {
    "file_purpose": "TikTok autocomplete and related-search expansion for an initial seed term.",
    "search_term": seed,
    "max_depth": MAX_DEPTH,
    "collected_at": utc_now_iso(),
    "layers": {str(depth): layers.get(depth, []) for depth in range(1, MAX_DEPTH + 1)},
    "branches": branches,
    "records": records,
    "edges": edges,
}
save_json(latest_json_path, payload)

print(f"Exported review CSV: {latest_csv_path}")
print(f"Exported nested JSON: {latest_json_path}")
for depth in range(1, MAX_DEPTH + 1):
    print(f"Order {depth}: {len(layers.get(depth, []))} terms")

## 6. Close Chrome

Run this after exporting. The dedicated profile remains available for later runs.

In [ ]:
await browser_context.close()
await playwright_instance.stop()
print("Browser closed.")

## Interpretation

First-order terms are the closest TikTok-native expansions of the seed. Second-order terms expose common modifiers, adjacent language and early branching. Third-order terms can reveal broader communities and topics, but they also have the greatest risk of semantic drift.

Every row includes `first_order_term` and `query_path`, so a second-order result remains visibly attached to the first-order branch that generated it. The JSON export also contains a `branches` collection with the same grouping.

Use `review_decision` for human triage: `analyse_videos` for terms worth passing to video discovery, `exclude` for clear semantic drift, and `uncertain` when a small manual TikTok check is needed. Record the reason in `review_notes`. The notebook deliberately does not infer these decisions from wording alone.